# 00 — Carte Hanoï + points de mesure

Ce notebook montre comment :
1. Télécharger la carte d'un quartier de Hanoï depuis OpenStreetMap
2. Afficher les routes sur une carte interactive
3. Ajouter des points de mesure (mock pour l'instant, réels plus tard)

In [ ]:
import osmnx as ox
import folium

# Télécharge le réseau routier depuis OpenStreetMap
# Essaie aussi : 'Bach Khoa, Hanoi', 'Hai Ba Trung, Hanoi'
G = ox.graph_from_place('Hoan Kiem, Hanoi, Vietnam', network_type='drive')
nodes, edges = ox.graph_to_gdfs(G)

print(f'{len(edges)} segments de route téléchargés')
print(f'{len(nodes)} intersections')

In [ ]:
# Crée la carte centrée sur la zone
center = [nodes.geometry.y.mean(), nodes.geometry.x.mean()]
m = folium.Map(location=center, zoom_start=15, tiles='CartoDB positron')

# Trace chaque segment de route
for _, row in edges.iterrows():
    coords = [(y, x) for x, y in row.geometry.coords]
    folium.PolyLine(coords, color='#555555', weight=1.5, opacity=0.6).add_to(m)

m

## Ajouter les points de mesure

Pour l'instant les valeurs dB sont inventées (mock).
Quand vous aurez vos vraies mesures terrain, remplacez les valeurs dans la liste.

In [ ]:
# Tes points de mesure — remplace par tes vraies données après le terrain
# Format : lat, lon, dB mesuré, description du lieu
points = [
    {'lat': 21.0285, 'lon': 105.8522, 'dB': 78, 'lieu': 'Bord route Đinh Tiên Hoàng'},
    {'lat': 21.0310, 'lon': 105.8490, 'dB': 54, 'lieu': 'Ruelle résidentielle'},
    # Ajoute tes points ici :
    # {'lat': ..., 'lon': ..., 'dB': ..., 'lieu': '...'},
]

# Couleur selon le niveau de bruit (seuils OMS)
def couleur(dB):
    if dB < 55:   return 'green'   # acceptable
    if dB < 70:   return 'orange'  # gênant
    return 'red'                   # dangereux

for p in points:
    # Cercle coloré
    folium.CircleMarker(
        location=[p['lat'], p['lon']],
        radius=12,
        color=couleur(p['dB']),
        fill=True,
        fill_opacity=0.85,
        popup=folium.Popup(f"<b>{p['lieu']}</b><br>{p['dB']} dB", max_width=200),
        tooltip=f"{p['dB']} dB"
    ).add_to(m)

    # Label flottant au-dessus du point
    folium.Marker(
        location=[p['lat'], p['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:bold;color:white;'
                 f'background:{couleur(p["dB"])};padding:2px 5px;'
                 f'border-radius:4px">{p["dB"]} dB</div>',
            icon_size=(60, 20),
            icon_anchor=(30, -8)
        )
    ).add_to(m)

m.save('../outputs/maps/hanoi_preview.html')
print('Carte sauvegardée → outputs/maps/hanoi_preview.html')
m

## Légende des couleurs

| Couleur | Niveau | Référence OMS |
|---|---|---|
| Vert | < 55 dB | Acceptable |
| Orange | 55–70 dB | Gênant, impact sur le sommeil |
| Rouge | > 70 dB | Dangereux, risque auditif |